In [2]:
from threading import Thread
from uuid import uuid4

from kombu import Connection

BROKER = "memory://"
episode = uuid4().hex  # Fresh inboxes each time the notebook cell runs.


def run_player(name, handle_message):
    with Connection(BROKER) as conn:
        inbox = conn.SimpleQueue(f"{episode}.{name}")
        try:
            while True:
                message = inbox.get(block=True)
                entry = message.payload
                print(f"entry: {entry}")
                if entry["kind"] == "stop":
                    message.ack()
                    return

                handle_message(entry)
                message.ack()
        finally:
            inbox.close()


def publish(conn, recipient, entry):
    inbox = conn.SimpleQueue(f"{episode}.{recipient}")
    try:
        inbox.put(entry, serializer="json")
    finally:
        inbox.close()

In [3]:
def ada_handler(entry):
    print("Ada received:", entry["body"])


def ben_handler(entry):
    print("Ben received:", entry["body"])


workers = [
    Thread(target=run_player, args=("ada", ada_handler)),
    Thread(target=run_player, args=("ben", ben_handler)),
]
for worker in workers:
    worker.start()

with Connection(BROKER) as conn:
    # Broadcast the initial task to both inboxes.
    for name in ("ada", "ben"):
        publish(conn, name, {"kind": "task", "body": "Investigate the samples."})

    # Deliver a directed message to Ben alone.
    publish(conn, "ben", {"kind": "message", "body": "Ada: do you have the instrument?"})

    # In the real runner, send these after a verified completion report.
    for name in ("ada", "ben"):
        publish(conn, name, {"kind": "stop"})

for worker in workers:
    worker.join()

entry: {'kind': 'task', 'body': 'Investigate the samples.'}entry: {'kind': 'task', 'body': 'Investigate the samples.'}
Ada received: Investigate the samples.
entry: {'kind': 'stop'}

Ben received: Investigate the samples.
entry: {'kind': 'message', 'body': 'Ada: do you have the instrument?'}
Ben received: Ada: do you have the instrument?
entry: {'kind': 'stop'}
